<a href="https://colab.research.google.com/github/carlosno/residencia-ia-sentinelas/blob/main/Extra%C3%A7%C3%A3otextosiguais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import pandas as pd
print("Baixando e preparando dados...")
url = "https://raw.githubusercontent.com/carlosno/residencia-ia-sentinelas/main/data/base_com_features.xlsx"
df = pd.read_excel(url)
print("Dados baixados com sucesso!")

Baixando e preparando dados...
Dados baixados com sucesso!


### Pré-processamento: Substituindo Entidades por Placeholders

Para melhorar a detecção de similaridade ignorando variações específicas como nomes e números, vamos criar uma função para substituir essas entidades por placeholders genéricos. Isso fará com que o `TfidfVectorizer` se concentre mais na estrutura e no conteúdo principal da mensagem.

In [2]:
import re

def preprocess_text_for_similarity(text):
    # Converte para string para garantir que a regex funcione
    text = str(text)
    # Substitui links/URLs
    text = re.sub(r'https?://\S+|www\.\S+', '[URL_PLACEHOLDER]', text)
    # Substitui emails
    text = re.sub(r'\S+@\S+', '[EMAIL_PLACEHOLDER]', text)
    # Substitui números de telefone (formato flexível)
    text = re.sub(r'\(?\d{2}\)?\s?\d{4,5}-?\d{4}', '[PHONE_PLACEHOLDER]', text)
    # Substitui @usernames (como em redes sociais)
    text = re.sub(r'@\w+', '[USERNAME_PLACEHOLDER]', text)
    # Substitui sequências de letras maiúsculas que podem ser nomes ou siglas
    # (Cuidado: pode pegar outras coisas, ajustar conforme a necessidade)
    text = re.sub(r'\b[A-Z][a-záàâãéèêíïóôõúüçñ\s]*[A-Z]\b', '[NAME_PLACEHOLDER]', text)
    # Substitui sequências de números (idades, valores, etc.)
    text = re.sub(r'\b\d+\s*(?:anos|reais|milhões|bilhões)?\b', '[NUMBER_PLACEHOLDER]', text)

    # Você pode adicionar mais regras de substituição conforme a necessidade
    return text


In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

global df_limpo # Make df_limpo accessible globally
global df_specific_string_removed_for_report # Make this accessible globally

# 1. Prepare the DataFrame
# Remove rows without text or ID and create a clean copy
df_limpo = df.dropna(subset=["id", "texto_tratado"]).copy()

# Reset the DataFrame's internal index for clean processing
df_limpo.reset_index(drop=True, inplace=True)
# The 'id' column for processing and matching results should now point to this internal index
df_limpo['id'] = df_limpo.index # This column will be used for iloc based results

df_specific_string_removed_for_report = pd.DataFrame() # Initialize as empty DataFrame

# --- INÍCIO: Adição para remover mensagens específicas solicitadas ---
target_string = "Você pode ativar o bot e começar a ganhar dinheiro aqui"
rows_containing_target = df_limpo[df_limpo['texto_tratado'].str.contains(target_string, na=False, case=False)]

if not rows_containing_target.empty:
    if len(rows_containing_target) > 1:
        # Keep the first occurrence (based on the current internal index after reset)
        index_to_keep_internal = rows_containing_target.index[0]
        indices_to_drop_specific_internal = [idx for idx in rows_containing_target.index if idx != index_to_keep_internal]

        # Capture the full rows of the specific messages to be dropped before dropping them
        df_specific_string_removed_for_report = df_limpo.loc[indices_to_drop_specific_internal].copy()
        # Add a column to indicate these were specifically removed
        df_specific_string_removed_for_report['tipo_remocao'] = 'string_especifica_direta'

        df_limpo = df_limpo.drop(indices_to_drop_specific_internal).reset_index(drop=True)
        # Update 'id' column to reflect the new sequential indices after dropping
        df_limpo['id'] = df_limpo.index
        print(f"Removidas {len(indices_to_drop_specific_internal)} duplicatas da mensagem específica contendo '{target_string}'.")
    else:
        print(f"Apenas uma ocorrência da mensagem específica encontrada ('{target_string}'), nenhuma remoção adicional necessária.")
else:
    print(f"Nenhuma ocorrência da mensagem específica encontrada ('{target_string}').")
# --- FIM: Adição para remover mensagens específicas solicitadas ---

# 2. Vetorização do texto usando TF-IDF (analisa sequências de palavras/caracteres)
# Usamos analyzer='char_wb' e ngram_range para capturar variações de digitação e pontuação
# Alterado para 'char_wb' e ngram_range mais amplo para maior sensibilidade a variações.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 7))
tfidf_matrix = vectorizer.fit_transform(df_limpo["texto_tratado"].astype(str))

# 3. Calcular a matriz de similaridade por Cosseno
matriz_similaridade = cosine_similarity(tfidf_matrix)

# 4. Filtrar os pares com similaridade maior ou igual a 60%
# Usamos np.triu para pegar apenas a metade superior da matriz e evitar duplicados (A com B e B com A)
linhas, colunas = np.where(np.triu(matriz_similaridade, k=1) >= 0.60)

# 5. Mapear os resultados de volta para os IDs e Textos originais
resultados = []
for i, j in zip(linhas, colunas):
    id_a = df_limpo.iloc[i]["id"]
    texto_a = df_limpo.iloc[i]["texto_tratado"]

    id_b = df_limpo.iloc[j]["id"]
    texto_b = df_limpo.iloc[j]["texto_tratado"]

    porcentagem_simil = matriz_similaridade[i, j] * 100

    resultados.append(
        {
            "ID_A": id_a,
            "Texto_A": texto_a,
            "ID_B": id_b,
            "Texto_B": texto_b,
            "Similaridade": f"{porcentagem_simil:.2f}%"
        }
    )

# 6. Criar o DataFrame final com os pares similares
df_similares = pd.DataFrame(resultados)

# Exibir os resultados
if not df_similares.empty:
    print(f"Foram encontrados {len(df_similares)} pares parecidos:")
    print(df_similares.head(10)) # Mostra os 10 primeiros resultados

    # Salvar o relatório final em uma planilha Excel ou CSV
    df_similares.to_csv("mensagens_60_porcento_similares.csv", index=False)
    print("\nArquivo 'mensagens_60_porcento_similares.csv' gerado com sucesso!")
else:
    print("Nenhuma mensagem com mais de 50% de similaridade foi encontrada.")

Removidas 1377 duplicatas da mensagem específica contendo 'Você pode ativar o bot e começar a ganhar dinheiro aqui'.
Foram encontrados 23251 pares parecidos:
   ID_A                                            Texto_A  ID_B  \
0     0  🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...   591   
1     0  🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...   668   
2     2  E AGORA XANDÃO ?\nVai mandar prender o General...   215   
3     2  E AGORA XANDÃO ?\nVai mandar prender o General...   444   
4     2  E AGORA XANDÃO ?\nVai mandar prender o General...   460   
5     2  E AGORA XANDÃO ?\nVai mandar prender o General...   697   
6     2  E AGORA XANDÃO ?\nVai mandar prender o General...  2374   
7     2  E AGORA XANDÃO ?\nVai mandar prender o General...  2380   
8     2  E AGORA XANDÃO ?\nVai mandar prender o General...  2386   
9     2  E AGORA XANDÃO ?\nVai mandar prender o General...  2459   

                                             Texto_B Similaridade  
0  ```Você só vai paga va

In [ ]:
# Aplica a função de pré-processamento à coluna 'texto_tratado' antes de limpar
df_limpo['texto_tratado_processed'] = df_limpo['texto_tratado'].apply(preprocess_text_for_similarity)

# 2. Vetorização do texto usando TF-IDF (analisa sequências de palavras/caracteres)
# Usamos analyzer='char_wb' e ngram_range para capturar variações de digitação e pontuação
# Alterado para 'char_wb' e ngram_range mais amplo para maior sensibilidade a variações.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 7))
tfidf_matrix = vectorizer.fit_transform(df_limpo["texto_tratado_processed"].astype(str))

# 3. Calcular a matriz de similaridade por Cosseno
matriz_similaridade = cosine_similarity(tfidf_matrix)

# 4. Filtrar os pares com similaridade maior ou igual a 70%
# Usamos np.triu para pegar apenas a metade superior da matriz e evitar duplicados (A com B e B com A)
# Correção: 'matriz_similarity' deve ser 'matriz_similaridade'
linhas, colunas = np.where(np.triu(matriz_similaridade, k=1) >= 0.70)

# 5. Mapear os resultados de volta para os IDs e Textos originais
resultados = []
for i, j in zip(linhas, colunas):
    id_a = df_limpo.iloc[i]["id"]
    texto_a = df_limpo.iloc[i]["texto_tratado"]

    id_b = df_limpo.iloc[j]["id"]
    texto_b = df_limpo.iloc[j]["texto_tratado"]

    porcentagem_simil = matriz_similaridade[i, j] * 100

    resultados.append(
        {
            "ID_A": id_a,
            "Texto_A": texto_a,
            "ID_B": id_b,
            "Texto_B": texto_b,
            "Similaridade": f"{porcentagem_simil:.2f}%"
        }
    )

# 6. Criar o DataFrame final com os pares similares
df_similares = pd.DataFrame(resultados)

# Exibir os resultados
if not df_similares.empty:
    print(f"Foram encontrados {len(df_similares)} pares parecidos:")
    print(df_similares.head(25))  # Mostra os 10 primeiros resultados

    # Salvar o relatório final em uma planilha Excel ou CSV
    df_similares.to_csv("mensagens_70_porcento_similares.csv", index=False)
    print("\nArquivo 'mensagens_70_porcento_similares.csv' gerado com sucesso!")
else:
    print("Nenhuma mensagem com mais de 70% de similaridade foi encontrada.")

Foram encontrados 38620 pares parecidos:
    ID_A                                            Texto_A  ID_B  \
0      0  🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...   591   
1      0  🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...   668   
2      2  E AGORA XANDÃO ?\nVai mandar prender o General...   215   
3      2  E AGORA XANDÃO ?\nVai mandar prender o General...   444   
4      2  E AGORA XANDÃO ?\nVai mandar prender o General...   460   
5      2  E AGORA XANDÃO ?\nVai mandar prender o General...   697   
6      2  E AGORA XANDÃO ?\nVai mandar prender o General...  2374   
7      2  E AGORA XANDÃO ?\nVai mandar prender o General...  2380   
8      2  E AGORA XANDÃO ?\nVai mandar prender o General...  2386   
9      2  E AGORA XANDÃO ?\nVai mandar prender o General...  2459   
10     3  *ATAQUE DE LULA AO “MEI” CAUSA REVOLTA E PODE ...   157   
11     3  *ATAQUE DE LULA AO “MEI” CAUSA REVOLTA E PODE ...   271   
12     3  *ATAQUE DE LULA AO “MEI” CAUSA REVOLTA E PODE ...  2

### Teste de Limiar de Similaridade: 60%

In [ ]:
# 4. Filtrar os pares com similaridade maior ou igual a 60%
linhas_60, colunas_60 = np.where(np.triu(matriz_similaridade, k=1) >= 0.60)

# 5. Mapear os resultados de volta para os IDs e Textos originais
resultados_60 = []
for i, j in zip(linhas_60, colunas_60):
    id_a = df_limpo.iloc[i]["id"]
    texto_a = df_limpo.iloc[i]["texto_tratado"]

    id_b = df_limpo.iloc[j]["id"]
    texto_b = df_limpo.iloc[j]["texto_tratado"]

    porcentagem_simil = matriz_similaridade[i, j] * 100

    resultados_60.append(
        {
            "ID_A": id_a,
            "Texto_A": texto_a,
            "ID_B": id_b,
            "Texto_B": texto_b,
            "Similaridade": f"{porcentagem_simil:.2f}%"
        }
    )

# 6. Criar o DataFrame final com os pares similares para 60%
df_similares_60 = pd.DataFrame(resultados_60)

# Exibir os resultados
if not df_similares_60.empty:
    print(f"Foram encontrados {len(df_similares_60)} pares parecidos com similaridade >= 60%:")
    display(df_similares_60.head())  # Mostra os 5 primeiros resultados

    # Salvar o relatório final em uma planilha Excel ou CSV
    df_similares_60.to_csv("mensagens_60_porcento_similares.csv", index=False)
    print("\nArquivo 'mensagens_60_porcento_similares.csv' gerado com sucesso!")
else:
    print("Nenhuma mensagem com mais de 60% de similaridade foi encontrada.")

Foram encontrados 132392 pares parecidos com similaridade >= 60%:


,ID_A,Texto_A,ID_B,Texto_B,Similaridade
0,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,591,```Você só vai paga valor depois que enviamos ...,98.40%
1,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,668,*Confiança total eu me responsabilizo ao você ...,99.39%
2,1,💲🔥 *URGENTE* 🔥💲\n💯 *NOSSO GRUPO DE REFERÊNCIA ...,970,ACUSAM O BOLSONARO DE GENOCIDA.\nSÓ LER E VERI...,60.09%
3,2,E AGORA XANDÃO ?\nVai mandar prender o General...,215,E AGORA XANDÃO ?\nVai mandar prender o General...,99.93%
4,2,E AGORA XANDÃO ?\nVai mandar prender o General...,444,E AGORA XANDÃO ?\nVai mandar prender o General...,99.73%



Arquivo 'mensagens_60_porcento_similares.csv' gerado com sucesso!


### Teste de Limiar de Similaridade: 50%

In [ ]:
# 4. Filtrar os pares com similaridade maior ou igual a 50%
linhas_50, colunas_50 = np.where(np.triu(matriz_similaridade, k=1) >= 0.50)

# 5. Mapear os resultados de volta para os IDs e Textos originais
resultados_50 = []
for i, j in zip(linhas_50, colunas_50):
    id_a = df_limpo.iloc[i]["id"]
    texto_a = df_limpo.iloc[i]["texto_tratado"]

    id_b = df_limpo.iloc[j]["id"]
    texto_b = df_limpo.iloc[j]["texto_tratado"]

    porcentagem_simil = matriz_similaridade[i, j] * 100

    resultados_50.append(
        {
            "ID_A": id_a,
            "Texto_A": texto_a,
            "ID_B": id_b,
            "Texto_B": texto_b,
            "Similaridade": f"{porcentagem_simil:.2f}%"
        }
    )

# 6. Criar o DataFrame final com os pares similares para 50%
df_similares_50 = pd.DataFrame(resultados_50)

# Exibir os resultados
if not df_similares_50.empty:
    print(f"Foram encontrados {len(df_similares_50)} pares parecidos com similaridade >= 50%:")
    display(df_similares_50.head())  # Mostra os 5 primeiros resultados

    # Salvar o relatório final em uma planilha Excel ou CSV
    df_similares_50.to_csv("mensagens_50_porcento_similares.csv", index=False)
    print("\nArquivo 'mensagens_50_porcento_similares.csv' gerado com sucesso!")
else:
    print("Nenhuma mensagem com mais de 50% de similaridade foi encontrada.")

Foram encontrados 534770 pares parecidos com similaridade >= 50%:


,ID_A,Texto_A,ID_B,Texto_B,Similaridade
0,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,13,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,54.78%
1,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,24,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,54.77%
2,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,25,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,54.92%
3,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,29,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,54.78%
4,0,🥳 *EMPRÉSTIMO SIMPLES E RÁPIDO SEM BUROCRACIA*...,43,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,55.02%



Arquivo 'mensagens_50_porcento_similares.csv' gerado com sucesso!


### Análise Comparativa de Amostras (60% vs 50% Similaridade)

In [ ]:
import pandas as pd

# Carregar os arquivos CSV
df_similares_60 = pd.read_csv('mensagens_60_porcento_similares.csv')
df_similares_50 = pd.read_csv('mensagens_50_porcento_similares.csv')

print("\n--- Amostra de 5 pares com Similaridade >= 60% ---")
display(df_similares_60.sample(min(15, len(df_similares_60))))

print("\n--- Amostra de 5 pares com Similaridade >= 50% ---")
display(df_similares_50.sample(min(15, len(df_similares_50))))


--- Amostra de 5 pares com Similaridade >= 60% ---


,ID_A,Texto_A,ID_B,Texto_B,Similaridade
122029,5156,"Feedback de Cláudio Borba 23 anos. Brazil, Suz...",5600,"Feedback de Filipe Franca 31 anos. Brazil, Nat...",66.30%
68684,3681,"Feedback de Emanuel Silva\n42 anos. Brasil, Ri...",4699,"Feedback de Henrique\n39 anos. Brasil, Mato Gr...",63.54%
103957,4645,"Feedback de Henrique Nunes 33 anos. Brazil, Ma...",5510,"Feedback de Felipe R.\n29 anos. Brasil, Pernam...",60.85%
116506,4932,"Feedback de Jonathan Sanches 30 anos. Brazil, ...",4952,"Feedback de Jonathan Sanches 29 anos. Brazil, ...",74.43%
5489,263,<URL>,372,<URL>,100.00%
89296,4231,"Feedback de Beatriz 34 anos de idade. Brazil, ...",5291,"Feedback de Marcela Oliveira 36 anos. Brasil, ...",62.23%
96865,4439,"feedback de Elena Ramos 37 anos. Brasil,Bahia\...",4774,"Feedback de Henrique Thiago 30 anos. Brasil, M...",61.88%
105615,4661,"Feedback de Henrique Thiago 30 anos. Brasil, M...",4807,"Feedback de Rodrigo\n34 anos. Brasil, Acre\nVo...",63.83%
26656,2448,*SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...,5751,INVISTA CONOSCO E RECEBA UM PAGAMENTO APÓS 36 ...,69.35%
113341,4836,"Feedback de Rocio Ferreira\n25 anos. Brasil, V...",4970,"Feedback de Afonso Rico 27 anos. Brazil, Limei...",60.65%



--- Amostra de 5 pares com Similaridade >= 50% ---


,ID_A,Texto_A,ID_B,Texto_B,Similaridade
452660,4848,"Feedback de Jorge Jardim 33 anos. Brazil, Baur...",6557,"Comentários de Leonardo B.\n36 anos. Brasil, T...",53.57%
102069,2949,"Feedback de Felipe\n31 anos. Brasil, Mato Gros...",4007,"Feedback de Miguel Cauã\n27 anos. Brasil, Mara...",53.78%
504468,5232,"Feedback de Carlos Lopes\n27 anos. Brazil, Car...",5304,"Feedback de Lucas Afonso 29 anos. Brasil, Rio ...",50.65%
461105,4907,R $ 1.000 para ganhar R $ 10.000\nR $ 1.500 pa...,5755,O sistema de negociação básico já está disponí...,72.36%
173699,3234,"Feedback de Nelson Alves\n23 anos. Brazil, Lim...",4083,Feedback de Nicolas De Barros\n25 anos. Brazil...,60.01%
301192,3948,"Feedback de Flavio\n36 anos. Brasil, Minas Ger...",5188,"Feedback de Marcos Gama\n23 anos. Brazil, Maca...",59.53%
291442,3897,"Feedback de Fernando Antunes 37 anos. Brazil, ...",4468,"Feedback de Marcos Gama\n23 anos. Brazil, Maca...",56.07%
66812,2741,"Feedback de Raphael Rijo 24 anos. Brazil, Baur...",4326,"Feedback de Luiz Miguel\n33 anos. Brasil, Rio ...",54.24%
300934,3948,"Feedback de Flavio\n36 anos. Brasil, Minas Ger...",4655,"Feedback de Henrique\n39 anos. Brasil, Mato Gr...",67.74%
206367,3396,"Feedback de Marcos Gama\n23 anos. Brazil, Maca...",3544,"Feedback de Roberto Franca 32 anos. Brazil, Vá...",50.46%


### Análise Quantitativa de Remoção para Diferentes Limiares

In [ ]:
def get_removed_indices_count(linhas_arr, colunas_arr):
    indices_para_remover = set()
    for i, j in zip(linhas_arr, colunas_arr):
        # Se a mensagem 'i' (primeira ocorrência) ainda não foi marcada para remoção,
        # significa que ela será a nossa "cópia oficial". A mensagem 'j' (segunda ocorrência) é a duplicada.
        if i not in indices_para_remover:
            indices_para_remover.add(j)
    return len(indices_para_remover)


# Número de mensagens removidas com 70% (já calculado na célula -ug5B3MxTjox)
# O valor de indices_para_remover foi 2762
removidos_70_porcento = 2762 # Valor obtido da execução anterior da célula -ug5B3MxTjox

# Calcular o número de mensagens removidas com 60%
removidos_60_porcento = get_removed_indices_count(linhas_60, colunas_60)

# Calcular o número de mensagens removidas com 50%
removidos_50_porcento = get_removed_indices_count(linhas_50, colunas_50)

print(f"Mensagens removidas com limiar de 70%: {removidos_70_porcento}")
print(f"Mensagens removidas com limiar de 60%: {removidos_60_porcento}")
print(f"Mensagens removidas com limiar de 50%: {removidos_50_porcento}")

print(f"\nSe o limiar fosse 60%, seriam removidas {removidos_60_porcento - removidos_70_porcento} mensagens adicionais.")
print(f"Se o limiar fosse 50%, seriam removidas {removidos_50_porcento - removidos_70_porcento} mensagens adicionais.")

Mensagens removidas com limiar de 70%: 2762
Mensagens removidas com limiar de 60%: 3122
Mensagens removidas com limiar de 50%: 3464

Se o limiar fosse 60%, seriam removidas 360 mensagens adicionais.
Se o limiar fosse 50%, seriam removidas 702 mensagens adicionais.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Use the globally available df_limpo that has already undergone specific string removal
# df_limpo = df.dropna(subset=["id", "texto_tratado"]).copy().reset_index(drop=True) # REMOVED

# 3. Identificar quais linhas devem ser excluídas
# np.triu garante que se A é igual a B, vamos olhar o par apenas uma vez
linhas, colunas = np.where(np.triu(matriz_similaridade, k=1) >= 0.50)

# Conjunto (set) para guardar os índices das mensagens duplicadas que vamos deletar
indices_para_remover = set()

for i, j in zip(linhas, colunas):
    # Se a mensagem 'i' (primeira ocorrência) ainda não foi marcada para remoção,
    # significa que ela será a nossa "cópia oficial". A mensagem 'j' (segunda ocorrência) é a duplicada.
    if i not in indices_para_remover:
        indices_para_remover.add(j)

# 4. Filtrar o DataFrame original excluindo os índices duplicados
# O .drop() remove as linhas que guardamos no conjunto
base_sem_duplicatas = df_limpo.drop(index=list(indices_para_remover))

# --- INÍCIO: Adição para converter colunas booleanas para 0 e 1 ---
# Lista das colunas a serem convertidas
boolean_cols = ['tem_url', 'tem_telefone', 'tem_email']

for col in boolean_cols:
    if col in base_sem_duplicatas.columns:
        # Mapeia 'VERDADEIRO' para 1 e 'FALSO' para 0
        base_sem_duplicatas[col] = base_sem_duplicatas[col].map({'VERDADEIRO': 1, 'FALSO': 0}).fillna(0).astype(int)
    else:
        print(f"Aviso: Coluna '{col}' não encontrada na base final e não será convertida.")
# --- FIM: Adição para converter colunas booleanas para 0 e 1 ---

# 5. Salvar a base de dados final limpa
base_sem_duplicatas.to_csv("base_limpa_sem_duplicatas.csv", index=False)

# Relatório do processo no terminal
print(f"--- Processo de Limpeza Concluído ---")
print(f"Tamanho original da base (após remoção de string específica): {len(df_limpo)} linhas")
print(f"Total de mensagens duplicadas removidas (por similaridade): {len(indices_para_remover)}")
print(f"Tamanho da nova base limpa: {len(base_sem_duplicatas)} linhas")
print(
    f"\nArquivo final salvo com sucesso como: 'base_limpa_sem_duplicatas.csv'"
)


--- Processo de Limpeza Concluído ---
Tamanho original da base: 7206 linhas
Total de mensagens duplicadas removidas: 3122
Tamanho da nova base limpa: 4084 linhas

Arquivo final salvo com sucesso como: 'base_limpa_sem_duplicatas.csv'


### Entradas Duplicadas Removidas

Agora vamos criar um arquivo contendo as entradas que foram identificadas como duplicadas e, por isso, removidas da base final `base_limpa_sem_duplicatas.csv`.

In [ ]:
# Criar um DataFrame com as entradas que foram removidas por similaridade
df_duplicatas_removidas_similaridade = df_limpo.loc[list(indices_para_remover)].copy()
df_duplicatas_removidas_similaridade['tipo_remocao'] = 'similaridade'

# Concatenar as mensagens removidas por similaridade com as removidas por string específica
df_duplicatas_removidas_final = pd.concat([
    df_duplicatas_removidas_similaridade,
    df_specific_string_removed_for_report # This global variable is from nc4HBP8eQyvx
], ignore_index=True)

# Salvar o DataFrame de todas as duplicatas em um novo arquivo CSV
df_duplicatas_removidas_final.to_csv("mensagens_duplicadas_removidas.csv", index=False)

print(f"Foram salvas {len(df_duplicatas_removidas_final)} mensagens duplicadas no arquivo 'mensagens_duplicadas_removidas.csv'.")
print(f"As primeiras 5 linhas das mensagens duplicadas são:\n{df_duplicatas_removidas_final.head(100)}")


Foram salvas 2491 mensagens duplicadas no arquivo 'mensagens_duplicadas_removidas.csv'.
As primeiras 5 linhas das mensagens duplicadas são:
                                         texto_tratado  golpe     classe  \
24   *SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...      0  nao_golpe   
25   *SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...      0  nao_golpe   
29   *SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...      0  nao_golpe   
34   <URL>\n*BRISANET CONECTANDO O MUNDO ATÉ VOCÊ\n...      0  nao_golpe   
43   *SE ALGUÉM TE PERGUNTAR O QUE FOI QUE BOLSONAR...      0  nao_golpe   
..                                                 ...    ...        ...   
536                                              <URL>      0  nao_golpe   
539  Ganhe 12.000.000 Kwai Golds semanalmente! De s...      0  nao_golpe   
542                                              <URL>      0  nao_golpe   
548  <URL>\n*BRISANET CONECTANDO O MUNDO ATÉ VOCÊ\n...      0  nao_golpe   
551  Perguntas simples p